In [ ]:
from xmlrpc.server import SimpleXMLRPCServer, SimpleXMLRPCRequestHandler
import xmlrpc.client
import time
from collections import defaultdict

class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

SERVER_URL  = "http://localhost:8008"
STATS_URL   = "http://localhost:8018"
RATE_LIMIT  = 5
RATE_PERIOD = 5
PROXY_PORT  = 8009

request_times = defaultdict(list)

server = SimpleXMLRPCServer(("localhost", PROXY_PORT), requestHandler=RequestHandler)
server.register_introspection_functions()

def add_log(event_type, duration=None):
    try:
        stats = xmlrpc.client.ServerProxy(STATS_URL,allow_none=True)
        stats.add_log(event_type, duration)
    except Exception as e:
        print(f"STATS ERROR: {e}")
    return True

def proxy_call(method_name, *args):
    client_ip = "127.0.0.1"
    now = time.time()

    request_times[client_ip] = [t for t in request_times[client_ip]
                                if now - t < RATE_PERIOD]
    if len(request_times[client_ip]) >= RATE_LIMIT:
        add_log('too_many_requests')
        raise Exception("Rate limit exceeded. Try again later.")
    request_times[client_ip].append(now)

    start = time.time()
    try:
        srv = xmlrpc.client.ServerProxy(SERVER_URL, allow_none=True)
        method = getattr(srv, method_name)
        result = method(*args)
    except Exception as e:
        add_log(f'{method_name}_error')
        raise e
    duration = time.time() - start

    add_log(method_name, duration)
    return result

methods_to_proxy = [
    'ping', 'now', 'type', 'sum', 'pow',
    'black_list_check', 'cloud_percent', 'color_inversion',
    'binarize_image', 'flip_vertical'
]

for m in methods_to_proxy:
    def make_proxy(name):
        def wrapper(*args):
            return proxy_call(name, *args)
        return wrapper
    server.register_function(make_proxy(m), m)

def proxy_ping():
    return "proxy_ok"
server.register_function(proxy_ping, 'proxy_ping')

print(f"Proxy on port {PROXY_PORT}...")
server.serve_forever()

Proxy on port 8009...


127.0.0.1 - - [19/Oct/2026 18:40:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:40:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:40:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:40:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:40:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:40:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:40:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:40:54] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:40:54] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:40:54] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:40:54] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:41:04] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:41:04] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:41:04] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 18:41:05] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/202